In [ ]:
# Pin dung bo phien ban da khoa trong uv.lock cua repo de lan chay sau tai lap duoc.
# Luu y: ban ke hoach pin sentence-transformers==3.0.1 / transformers==4.44.2 /
# faiss-cpu==1.8.0 nhung bo do khong chay duoc voi Qwen3-Embedding: transformers
# chi dang ky kien truc Qwen3 tu 4.51, con pyproject.toml cua repo yeu cau
# sentence-transformers>=3.4 va faiss-cpu>=1.9. Ba pin duoi day lay dung tu uv.lock.
# torch dung san cua Colab (khong cai lai -- ban pip rieng se rat nang).
!pip install -q "sentence-transformers==5.6.1" "transformers==5.14.1" "faiss-cpu==1.14.3"
!nvidia-smi

In [ ]:
# Mount Drive + clone repo.
# Chuan bi TRUOC tren Drive (data/raw, data/interim, data/processed bi .gitignore
# nen clone GitHub khong co): chep snapshot tu may local len
#   MyDrive/vifinqa/repo-data/processed/release_v2_422df141c935/  (manifest.json + 3 parquet)
#   MyDrive/vifinqa/repo-data/interim/week1_gate/422df141c935/gate-result.json
import shutil
import sys
from pathlib import Path

from google.colab import drive

REPO = Path("/content/repo")
DRIVE_DATA = Path("/content/drive/MyDrive/vifinqa/repo-data")

drive.mount("/content/drive")
if not (REPO / ".git").exists():
    !git clone https://github.com/TranVu2005/financial-assistant /content/repo
%cd /content/repo

# requires-python ">=3.11,<3.12" cua repo bi chan tren Python moi hon cua Colab,
# nen thay `pip install -e .` bang cach tro thang sys.path vao src layout;
# cach nay con giu nguyen pin o Cell 1 (khong bi pip nang cap de len).
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

SNAPSHOT = {
    "processed/release_v2_422df141c935": "data/processed/release_v2_422df141c935",
    "interim/week1_gate/422df141c935/gate-result.json": "data/interim/week1_gate/422df141c935/gate-result.json",
}
for source_relative, target_relative in SNAPSHOT.items():
    target = REPO / target_relative
    if target.exists():
        continue  # da sao chep o lan chay truoc
    source = DRIVE_DATA / source_relative
    if not source.exists():
        raise FileNotFoundError(f"Missing {source}; upload it from local before running")
    target.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, target)
    else:
        shutil.copy2(source, target)
    print("copied:", target_relative)

In [ ]:
# Dung corpus (chay tren CPU, nhanh).
# API that: build_dense_corpus nhan tuple documents + dataset_fingerprint +
# release_lock_sha256 (khong nhan object release nhu trong ban ke hoach);
# documents duoc dung bang build_table_documents tren ba parquet cua release.
from pathlib import Path

from financial_report_qa.retrieval.dense_corpus import build_dense_corpus, save_dense_corpus
from financial_report_qa.retrieval.documents import build_table_documents
from financial_report_qa.retrieval.release import resolve_retrieval_release

release = resolve_retrieval_release(
    Path("data/qa/week1_pilot_422df141c935/dataset-pilot-v1.json"),
    repo_root=Path.cwd(),
)
corpus = build_dense_corpus(
    build_table_documents(
        release.release_dir / "documents.parquet",
        release.release_dir / "tables.parquet",
        release.release_dir / "cells.parquet",
    ),
    dataset_fingerprint=release.dataset_fingerprint,
    release_lock_sha256=release.lock_sha256,
)
save_dense_corpus(corpus, Path("/content/drive/MyDrive/vifinqa/dense-corpus"))
print(len(corpus.documents), "documents")

In [ ]:
# Embed theo shard, checkpoint tung shard (session timeout khong mat viec).
import os

# Phai dat truoc khi torch khoi tao cuBLAS: dense_encoder.py bat
# torch.use_deterministic_algorithms(True) cho CUDA va yeu cau bien nay tu truoc.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
# Cache model ~8GB tren Drive de session moi khong tai lai tu dau.
os.environ.setdefault("HF_HOME", "/content/drive/MyDrive/vifinqa/hf-cache")

from pathlib import Path

import numpy as np

from financial_report_qa.retrieval.dense_encoder import (
    SentenceTransformerDenseEncoder,
    approved_encoder_spec,
)

SHARD = 2048
out = Path("/content/drive/MyDrive/vifinqa/qwen3-shards")
out.mkdir(parents=True, exist_ok=True)

spec = approved_encoder_spec("qwen3-embedding-4b").model_copy(update={"device": "cuda"})
encoder = SentenceTransformerDenseEncoder(spec)
texts = [document.text for document in corpus.documents]

for start in range(0, len(texts), SHARD):
    target = out / f"shard_{start:07d}.npy"
    if target.exists():
        continue  # da xong o lan chay truoc -- bo qua, khong tinh lai
    vectors = encoder.encode_documents(texts[start : start + SHARD])
    # Ghi qua file tam roi rename nguyen tu: shard do dang do ngat giua chung
    # se khong lot qua phep kiem tra target.exists() o lan chay sau.
    # N5: giu float32 khi ghi ra dia -- KHONG doi dtype. Neu VRAM T4 khong du,
    # giam SHARD (don bay chinh vi ST tu chia batch noi bo), van giu float32.
    partial = out / f".{target.name}.partial"
    with partial.open("wb") as stream:
        np.save(stream, vectors.astype(np.float32))
    partial.replace(target)
    print(target.name, vectors.shape, flush=True)

In [ ]:
# Ghep shard va dung FAISS index.
# API that: build_dense_index(corpus, encoder) tu encode tung batch; de dung
# dung vector da embed theo shard (tiet kiem nhieu gio GPU), boc chung trong
# mot adapter dung giao dien DenseEncoder thay vi truyen mang vector.
from pathlib import Path

import numpy as np

from financial_report_qa.retrieval.dense_index import build_dense_index, save_dense_index


class ShardVectorsEncoder:
    """DenseEncoder adapter: tra lai vector da embed san theo dung thu tu corpus."""

    def __init__(self, spec, vectors):
        self.spec = spec
        self._vectors = vectors
        self._cursor = 0

    def encode_documents(self, texts):
        texts = tuple(texts)
        chunk = self._vectors[self._cursor : self._cursor + len(texts)]
        if len(chunk) != len(texts):
            raise ValueError("shard vectors exhausted before all corpus documents")
        self._cursor += len(texts)
        return chunk

    def encode_query(self, text):
        raise NotImplementedError("index build does not encode queries")


shards = sorted(Path("/content/drive/MyDrive/vifinqa/qwen3-shards").glob("shard_*.npy"))
vectors = np.concatenate([np.load(path) for path in shards], axis=0).astype(np.float32)
assert vectors.shape[0] == len(corpus.documents), (vectors.shape[0], len(corpus.documents))
assert vectors.shape[1] == 2560, vectors.shape

# device quay ve "cpu" nhu approved spec goc: encoder_spec_sha256 ghi trong
# manifest khop voi phia local (CLI mac dinh --encoder-device cpu) khi
# load_dense_index kiem tra identity. Adapter khong chay model nen device
# chi la nhan trong spec.
builder_spec = spec.model_copy(update={"device": "cpu"})
index = build_dense_index(corpus, ShardVectorsEncoder(builder_spec, vectors))
save_dense_index(index, Path("/content/drive/MyDrive/vifinqa/dense-qwen3-4b"))
print(index.manifest.document_count, index.manifest.dimension, index.manifest.encoder_spec_sha256)

In [ ]:
# Kiem tra tinh toan ven truoc khi tai ve.
# API that: load_dense_index can corpus + expected_encoder_spec_sha256 +
# release_lock_sha256 de doi chieu identity (khong phai goi mot tham so).
from pathlib import Path

from financial_report_qa.retrieval.dense_encoder import (
    approved_encoder_spec,
    encoder_spec_sha256,
)
from financial_report_qa.retrieval.dense_index import load_dense_index

loaded = load_dense_index(
    Path("/content/drive/MyDrive/vifinqa/dense-qwen3-4b"),
    corpus,
    expected_encoder_spec_sha256=encoder_spec_sha256(approved_encoder_spec("qwen3-embedding-4b")),
    release_lock_sha256=release.lock_sha256,
)
print(
    loaded.manifest.document_count,
    loaded.manifest.dimension,
    loaded.manifest.encoder.name,
    loaded.manifest.dataset_fingerprint,
)
assert loaded.manifest.document_count == len(corpus.documents)
assert loaded.manifest.dimension == 2560
assert loaded.manifest.dataset_fingerprint == release.dataset_fingerprint